# Databento from csv to Parquet Data Catalog

This notebook prepares historical market data exported from databento into a Nautilus Trader-compatible Parquet data catalog. It covers input validation, schema mapping (symbols, venues, timestamps, prices, volumes), normalization (timezones, dtypes), and writing partitioned Parquet files organized as a data catalog ready for ingestion by Nautilus Trader.

In [1]:
import pandas as pd
# imports
from nautilus_trader.adapters.databento import DatabentoDataLoader
from nautilus_trader.data.engine import ParquetDataCatalog
from nautilus_trader.model import BarType, Bar
from nautilus_trader.persistence.wranglers import BarDataWrangler
from nautilus_trader.test_kit.providers import TestInstrumentProvider

In [2]:
# Load consolidated MBP-1 quotes
loader = DatabentoDataLoader()
instrument = TestInstrumentProvider.es_future(2025, 12)

csv_file_path = r"../data/glbx-mdp3-20201020-20251019.ohlcv-1m_higher_volume.csv"
TF = pd.Timedelta(minutes=1)   # 1M timeframe

df = pd.read_csv(csv_file_path, sep=",", decimal=".", header=0, index_col=False)

In [3]:
open_ts = pd.to_datetime(
    df['ts_event'],
    utc=True,
)
close_ts = open_ts + TF      # bar close time

In [4]:
df['timestamp'] = close_ts
df = df.set_index('timestamp').sort_index()

In [5]:
# Keep only required columns
df = df[['open', 'high', 'low', 'close', 'volume']]

catalog = ParquetDataCatalog("../catalog")
catalog.write_data([instrument])  # Store instrument definition(s)

# Prepare 1-minute BarType for the instrument
ES_FUTURES_1_MIN_BAR_TYPE = BarType.from_str(f"{instrument.id}-1-MINUTE-LAST-EXTERNAL")

# `BarDataWrangler` converts each row into objects of the type `Bar`
wrangler = BarDataWrangler(ES_FUTURES_1_MIN_BAR_TYPE, instrument)
wrangled_bars_1_min: list[Bar] = wrangler.process(df)#, ts_init_delta=int(TF.total_seconds() * 1e9))

# Write bars data to the catalog
catalog.write_data(wrangled_bars_1_min)  # Store bar data